In [1]:
from google.colab import drive
drive.mount('/content/drive')
import shutil, os
shutil.copytree("/content/drive/MyDrive/dppo", "/content/dppo", dirs_exist_ok=True)
os.chdir("/content/dppo")
print(os.listdir("."))  # should show your .py files
!pip install diffusers gymnasium gymnasium-robotics mujoco imageio[ffmpeg]
!pip install minari

: 

In [ ]:
import sys
import os
import glob

dppo_path = '/content/dppo'
if dppo_path not in sys.path:
    sys.path.insert(0, dppo_path)
elif sys.path[0] != dppo_path:
    sys.path.remove(dppo_path)
    sys.path.insert(0, dppo_path)

import torch
from copy import deepcopy
from diffusers import DDPMScheduler

from network import DiffusionMLPActor, ValueCritic
from dppo_math import DashcamBuffer
from train import train_behavior_cloning, train_dppo
from wrappers import make_dppo_env
import gymnasium_robotics
import gymnasium as gym
gym.register_envs(gymnasium_robotics)

# ── Config ────────────────────────────────────────────────────────────────────
ENV_ID        = "AdroitHandPen-v1"
OBS_DIM       = 45
ACT_DIM       = 24
CHUNK_SIZE    = 4
EXECUTE_TA    = 2
K             = 100
K_PRIME       = 3
NUM_ENV_STEPS = 128
HIDDEN_DIM    = 256

BC_EPOCHS     = 50
BC_BATCH_SIZE = 256

SAVE_DIR = "/content/drive/MyDrive/dppo/checkpoints"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
# ──────────────────────────────────────────────────────────────────────────────

def main():
    os.chdir('/content/dppo')
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Running on: {DEVICE}")

    # ── Load expert normalization stats ──────────────────────────────────────
    expert_data_path = "expert_data.pt"
    obs_mean = obs_std = None
    if os.path.exists(expert_data_path):
        _stats = torch.load(expert_data_path, map_location="cpu")
        if "obs_mean" in _stats:
            obs_mean = _stats["obs_mean"].numpy()
            obs_std  = _stats["obs_std"].numpy()
            print(f"Loaded fixed obs normalization stats from {expert_data_path}")

    # ── Environment ──────────────────────────────────────────────────────────
    env = make_dppo_env(ENV_ID, Ta=EXECUTE_TA, obs_mean=obs_mean, obs_std=obs_std)

    # ── Networks ─────────────────────────────────────────────────────────────
    actor  = DiffusionMLPActor(OBS_DIM, ACT_DIM, CHUNK_SIZE, HIDDEN_DIM).to(DEVICE)
    critic = ValueCritic(OBS_DIM, HIDDEN_DIM).to(DEVICE)

    # ── Diffusion scheduler ───────────────────────────────────────────────────
    scheduler = DDPMScheduler(
        num_train_timesteps=K,
        beta_schedule="squaredcos_cap_v2",
        clip_sample=True,
        prediction_type="epsilon",
    )

    # ── Rollout buffer ────────────────────────────────────────────────────────
    buffer = DashcamBuffer(
        num_env_steps=NUM_ENV_STEPS,
        K_prime=K_PRIME,
        obs_dim=OBS_DIM,
        chunk_size=CHUNK_SIZE,
        act_dim=ACT_DIM,
        device=DEVICE,
    )

    # ── Phase 1: Behavior Cloning ─────────────────────────────────────────────
    bc_ckpt = os.path.join(SAVE_DIR, "actor_bc.pt")
    if os.path.exists(bc_ckpt):
        actor.load_state_dict(torch.load(bc_ckpt, map_location=DEVICE))
        print("Loaded existing BC checkpoint — skipping pretraining.")
    elif os.path.exists(expert_data_path):
        data = torch.load(expert_data_path, map_location=DEVICE)
        expert_states = data["states"]
        expert_chunks = data["action_chunks"]
        print(f"Loaded {len(expert_states)} expert transitions. Starting BC pretraining...")
        actor = train_behavior_cloning(
            actor, scheduler, expert_states, expert_chunks,
            K=K, epochs=BC_EPOCHS, batch_size=BC_BATCH_SIZE, device=DEVICE,
        )
        torch.save(actor.state_dict(), bc_ckpt)
        print("BC pretraining done.")
    else:
        print("No expert_data.pt found — skipping BC, starting DPPO from random init.")

    # ── Resume DPPO checkpoint if available ──────────────────────────────────
    dppo_ckpts = sorted(glob.glob(os.path.join(SAVE_DIR, "actor_iter*.pt")))
    if dppo_ckpts:
        latest_actor  = dppo_ckpts[-1]
        latest_critic = latest_actor.replace("actor_iter", "critic_iter")
        actor.load_state_dict(torch.load(latest_actor, map_location=DEVICE))
        if os.path.exists(latest_critic):
            critic.load_state_dict(torch.load(latest_critic, map_location=DEVICE))
        print(f"Resumed DPPO from {latest_actor}")

    # ── Phase 2: DPPO ─────────────────────────────────────────────────────────
    old_actor = deepcopy(actor).to(DEVICE)
    print("Starting DPPO fine-tuning...")
    train_dppo(
        env, old_actor, actor, critic, buffer, scheduler, K, K_PRIME,
        device=DEVICE,
        save_dir=SAVE_DIR,
        save_every=50,
        eval_every=20,
        num_eval_episodes=20,
    )

    # ── Save final checkpoints ────────────────────────────────────────────────
    torch.save(actor.state_dict(),  os.path.join(SAVE_DIR, "actor_final.pt"))
    torch.save(critic.state_dict(), os.path.join(SAVE_DIR, "critic_final.pt"))
    print(f"Training complete. All checkpoints saved to {SAVE_DIR}/")

    env.close()


if __name__ == "__main__":
    main()
